# Chongchen Jiang — Model Development and Validation
## 1. Validation Strategy: Initial Data Inspection

**Objective:** Examine the transaction structure and repeated
property sales before choosing a validation strategy.

In [1]:
import pandas as pd

# Load raw transaction data
sales = pd.read_csv("data/sales.csv")

# Inspect the dataset
print("Dataset shape:", sales.shape)
print("\nColumns:")
print(sales.columns.tolist())

print("\nData types:")
print(sales.dtypes)

# Check date coverage
sales["sale_date"] = pd.to_datetime(
    sales["sale_date"], errors="coerce"
)

print("\nTransactions by year:")
print(sales["sale_date"].dt.year.value_counts().sort_index())

# Check duplicate transactions
print("\nDuplicate transaction IDs:")
print(sales["transaction_id"].duplicated().sum())

# Check repeated property sales
property_counts = sales.groupby("property_id").size()

print("\nProperties with multiple sales:")
print((property_counts > 1).sum())

Dataset shape: (109307, 4)

Columns:
['property_id', 'transaction_id', 'sale_date', 'sale_price']

Data types:
property_id         int64
transaction_id      int64
sale_date          object
sale_price        float64
dtype: object

Transactions by year:
2023    37520
2024    35465
2025    36322
Name: sale_date, dtype: int64

Duplicate transaction IDs:
0

Properties with multiple sales:
9774


In [2]:

# Check property overlap between 2023 and 2024

train_2023 = sales[
    sales["sale_date"].dt.year == 2023
].copy()

val_2024 = sales[
    sales["sale_date"].dt.year == 2024
].copy()

# Properties observed in the training period
train_property_ids = set(train_2023["property_id"])

# Identify whether each validation transaction
# belongs to a previously observed property
val_2024["seen_property"] = (
    val_2024["property_id"].isin(train_property_ids)
)

n_seen = val_2024["seen_property"].sum()
n_total = len(val_2024)
n_unseen = n_total - n_seen

print("2024 validation transactions:", n_total)
print("Seen-property transactions:", n_seen)
print("Unseen-property transactions:", n_unseen)
print("Seen-property proportion:", round(n_seen / n_total * 100, 2), "%")


2024 validation transactions: 35465
Seen-property transactions: 2498
Unseen-property transactions: 32967
Seen-property proportion: 7.04 %


In [3]:

# Compare sale-price distributions between
# seen and unseen properties in 2024

price_comparison = (
    val_2024.groupby("seen_property")["sale_price"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90)
    )
)

price_comparison.index = price_comparison.index.map({
    False: "Unseen properties",
    True: "Seen properties"
})

display(price_comparison.round(2))


,count,mean,median,p25,p75,p90
seen_property,,,,,,
Unseen properties,32967,447977.75,345500.0,230000.0,515550.0,835000.0
Seen properties,2498,440161.06,331500.0,250000.0,460000.0,720000.0


In [4]:

# Global Median Baseline
# Train using 2023 prices only

global_median = train_2023["sale_price"].median()

# Predict the same value for all 2024 transactions
val_2024["baseline_pred"] = global_median

# Calculate absolute errors
val_2024["baseline_ae"] = (
    val_2024["sale_price"] - val_2024["baseline_pred"]
).abs()

# Overall performance
overall_mae = val_2024["baseline_ae"].mean()

# Seen vs Unseen performance
subgroup_mae = (
    val_2024.groupby("seen_property")["baseline_ae"]
    .agg(["count", "mean", "median"])
)

subgroup_mae.index = subgroup_mae.index.map({
    False: "Unseen properties",
    True: "Seen properties"
})

print("Training global median:", round(global_median, 2))
print("Overall validation MAE:", round(overall_mae, 2))
print("\nBaseline absolute errors by group:")
display(subgroup_mae.round(2))


Training global median: 315000.0
Overall validation MAE: 231085.44

Baseline absolute errors by group:


,count,mean,median
seen_property,,,
Unseen properties,32967,233138.77,130000.0
Seen properties,2498,203986.90,95000.0



### Baseline Subgroup Analysis

The Global Median Baseline produces the same prediction
for every property, regardless of property identity.

On the 2024 validation set, the MAE is $233,138.77 for
unseen properties and $203,986.90 for seen properties.

This difference cannot be attributed to the baseline
memorising property identities, because property IDs are
not used for prediction.

The result demonstrates that subgroup error differences
can arise even without property-specific learning.

Therefore, lower errors for previously seen properties
would not, by themselves, establish data leakage.
Further validation with more informative models is needed.


In [5]:

# Inspect property attributes before joining

properties = pd.read_csv("data/properties_2022.csv")

print("Property dataset shape:", properties.shape)

print("\nPossible location columns:")
print([
    col for col in properties.columns
    if "town" in col.lower()
    or "location" in col.lower()
    or "latitude" in col.lower()
    or "longitude" in col.lower()
])

print("\nDuplicate property IDs:")
print(properties["property_id"].duplicated().sum())


Property dataset shape: (98899, 98)

Possible location columns:
['loc_longitude', 'loc_latitude', 'meta_township_code']

Duplicate property IDs:
0


C:\Users\众神已死，我是太阳\AppData\Local\Temp\ipykernel_13524\3200835891.py:3: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  properties = pd.read_csv("data/properties_2022.csv")


In [6]:

# Check the sales-to-property join before building
# the Township Median Baseline

property_location = properties[
    ["property_id", "meta_township_code"]
].copy()

sales_with_location = sales.merge(
    property_location,
    on="property_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Original sales rows:", len(sales))
print("Rows after joining:", len(sales_with_location))

print("\nJoin status:")
print(sales_with_location["_merge"].value_counts())

print("\nMissing township codes:")
print(sales_with_location["meta_township_code"].isna().sum())

print("\nNumber of distinct townships:")
print(sales_with_location["meta_township_code"].nunique())

print("\nSample township codes:")
print(sales_with_location["meta_township_code"].dropna().unique()[:10])


Original sales rows: 109307
Rows after joining: 109307

Join status:
both          109307
left_only          0
right_only         0
Name: _merge, dtype: int64

Missing township codes:
0

Number of distinct townships:
38

Sample township codes:
[25 21 30 70 28 37 12 29 16 77]


In [9]:

import numpy as np
import pandas as pd

# --------------------------------------------------
# Research questions:
# 1. Does Mean or Median work better as a baseline?
# 2. Does adding Township information improve prediction?
#
# Preliminary split:
# Train: 2023
# Validation: 2024
# Final test: 2025 (not used here)
# --------------------------------------------------

df = sales_with_location.copy()
df["sale_date"] = pd.to_datetime(df["sale_date"])

train = df[df["sale_date"].dt.year == 2023].copy()
val = df[df["sale_date"].dt.year == 2024].copy()

# 1. Learn all baseline statistics from TRAIN only

global_mean = train["sale_price"].mean()
global_median = train["sale_price"].median()

township_stats = (
    train.groupby("meta_township_code")["sale_price"]
    .agg(["mean", "median"])
)

# 2. Predict on the SAME validation observations

val["pred_global_mean"] = global_mean
val["pred_global_median"] = global_median

val["pred_township_mean"] = (
    val["meta_township_code"]
    .map(township_stats["mean"])
    .fillna(global_mean)
)

val["pred_township_median"] = (
    val["meta_township_code"]
    .map(township_stats["median"])
    .fillna(global_median)
)

# 3. Evaluate with identical metrics

def evaluate_regression(y_true, y_pred):
    errors = np.asarray(y_true) - np.asarray(y_pred)

    return {
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2))
    }

baseline_columns = {
    "Global Mean": "pred_global_mean",
    "Global Median": "pred_global_median",
    "Township Mean": "pred_township_mean",
    "Township Median": "pred_township_median"
}

results = []

for name, prediction_col in baseline_columns.items():
    metrics = evaluate_regression(
        val["sale_price"],
        val[prediction_col]
    )

    results.append({
        "Baseline": name,
        **metrics
    })

baseline_results = pd.DataFrame(results).set_index("Baseline")

print("2023 training mean:", round(global_mean, 2))
print("2023 training median:", round(global_median, 2))

print("\nTownships in training:", len(township_stats))
print(
    "Validation observations in unseen townships:",
    (~val["meta_township_code"].isin(township_stats.index)).sum()
)

print("\n2024 Validation Results:")
display(baseline_results.round(2))


2023 training mean: 406135.87
2023 training median: 315000.0

Townships in training: 38
Validation observations in unseen townships: 0

2024 Validation Results:


,MAE,RMSE
Baseline,,
Global Mean,237022.94,425084.91
Global Median,231085.44,443316.10
Township Mean,170855.63,319593.58
Township Median,170953.95,337490.59


### Baseline Comparison — Preliminary Findings

Four baseline methods were compared using 2023 training data and the same 2024 validation set. The experiment varied two factors: the summary statistic (mean versus median) and the use of location information (global versus township-specific).

Township-based methods consistently outperformed their corresponding global methods in both MAE and RMSE, suggesting that township information provides substantial predictive value.

The Township Mean achieved the lowest validation MAE ($170,855.63) and RMSE ($319,593.58). However, its MAE advantage over Township Median was only $98.32, so the practical significance and stability of this small difference remain uncertain.

We provisionally retain Global Mean as a minimal benchmark and Township Mean as a location-aware benchmark. These results will be reassessed using the final validation procedure and cleaned dataset.

Training mean: 406135.87
Validation MAE: 237022.94
Validation RMSE: 425084.91
